# Website Traffic Seasonality Analysis

## Project Overview
Analyze web traffic over time to uncover trend, seasonality, campaign effects, and traffic anomalies.

## Learning Objectives
- Decompose web traffic into trend, seasonality, and irregular components
- Identify campaign-driven traffic spikes
- Analyze traffic by source, device, and day of week
- Detect anomalous traffic days

## Problem Statement
Digital marketing teams need to distinguish organic growth from campaign-driven spikes, understand weekly patterns, and detect anomalies that may indicate technical issues or bot traffic.

## Why This Project Matters
Web traffic is the lifeblood of digital business. Understanding its components enables better budget allocation, content scheduling, and capacity planning.

## Dataset Overview
Synthetic daily web traffic: 2 years (~730 days) with sessions, page views, bounce rate, traffic source, and device type.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_days = 730
dates = pd.date_range('2022-01-01', periods=n_days, freq='D')
t = np.arange(n_days)

# Trend + weekly + annual seasonality
trend = 5000 + t * 5
weekly = 800 * np.sin(2 * np.pi * t / 7)  # weekday peaks
annual = 600 * np.sin(2 * np.pi * (t - 60) / 365.25)
noise = np.random.normal(0, 400, n_days)

# Campaign spikes (random bursts)
campaigns = np.zeros(n_days)
campaign_days = [45, 120, 200, 310, 400, 500, 620]
for cd in campaign_days:
    if cd < n_days:
        campaigns[cd] = np.random.uniform(3000, 8000)
        if cd + 1 < n_days: campaigns[cd + 1] = campaigns[cd] * 0.5
        if cd + 2 < n_days: campaigns[cd + 2] = campaigns[cd] * 0.2

sessions = np.clip(trend + weekly + annual + campaigns + noise, 1000, None).round(0).astype(int)
pages_per_session = np.clip(np.random.normal(3.2, 0.5, n_days), 1.5, 6).round(2)
page_views = (sessions * pages_per_session).astype(int)
bounce_rate = np.clip(55 - t * 0.005 + np.random.normal(0, 3, n_days), 30, 75).round(1)

source = np.random.choice(['Organic', 'Paid', 'Social', 'Direct', 'Referral'], n_days,
                            p=[0.40, 0.20, 0.15, 0.15, 0.10])
device = np.random.choice(['Desktop', 'Mobile', 'Tablet'], n_days, p=[0.45, 0.45, 0.10])

df = pd.DataFrame({
    'Date': dates, 'Sessions': sessions, 'PageViews': page_views,
    'PagesPerSession': pages_per_session, 'BounceRate': bounce_rate,
    'TopSource': source, 'TopDevice': device
})
df = df.set_index('Date')
df['DayOfWeek'] = df.index.day_name()
df['Month'] = df.index.month
df['Year'] = df.index.year

print(f'Dataset: {df.shape}')
print(f'Sessions range: {df["Sessions"].min():,} — {df["Sessions"].max():,}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date gaps: {pd.date_range(df.index.min(), df.index.max()).difference(df.index).size}')

## Traffic Trend with Moving Averages

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['Sessions'], alpha=0.3, linewidth=0.5, label='Daily')
ax.plot(df.index, df['Sessions'].rolling(7).mean(), color='red', linewidth=1.5, label='7-day MA')
ax.plot(df.index, df['Sessions'].rolling(30).mean(), color='darkblue', linewidth=2, label='30-day MA')
ax.set_title('Daily Sessions with Moving Averages')
ax.set_ylabel('Sessions')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'traffic_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Decomposition

In [ ]:
decomp = seasonal_decompose(df['Sessions'], model='additive', period=7)
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Weekly Seasonality')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'decomposition.png'), dpi=100, bbox_inches='tight')
plt.show()

## Day-of-Week Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df.groupby('DayOfWeek')['Sessions'].mean().reindex(dow_order).plot.bar(ax=ax, edgecolor='black', color='steelblue')
ax.set_title('Average Sessions by Day of Week')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'day_of_week.png'), dpi=100, bbox_inches='tight')
plt.show()

## Campaign Spike Detection

In [ ]:
# Anomaly detection: sessions > mean + 2.5*std of rolling window
rolling_mean = df['Sessions'].rolling(14, center=True).mean()
rolling_std = df['Sessions'].rolling(14, center=True).std()
df['IsAnomaly'] = df['Sessions'] > (rolling_mean + 2.5 * rolling_std)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['Sessions'], alpha=0.5, linewidth=0.5)
ax.scatter(df.index[df['IsAnomaly']], df.loc[df['IsAnomaly'], 'Sessions'],
           color='red', s=40, zorder=5, label=f'Anomalies ({df["IsAnomaly"].sum()} days)')
ax.set_title('Traffic Anomalies (Campaign Spikes)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'anomalies.png'), dpi=100, bbox_inches='tight')
plt.show()

## Traffic Source and Device Mix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['TopSource'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Traffic Source Mix')
axes[0].set_ylabel('')
df['TopDevice'].value_counts().plot.pie(ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Device Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'source_device.png'), dpi=100, bbox_inches='tight')
plt.show()

## Bounce Rate Trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['BounceRate'], alpha=0.3, linewidth=0.5)
ax.plot(df.index, df['BounceRate'].rolling(30).mean(), color='coral', linewidth=2, label='30-day MA')
ax.set_title('Bounce Rate Trend')
ax.set_ylabel('Bounce Rate (%)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'bounce_rate.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Bounce rate improvement: {df["BounceRate"].iloc[:30].mean():.1f}% → {df["BounceRate"].iloc[-30:].mean():.1f}%')

## Interpretation and Business Insight
- **Strong upward trend** — sessions grow ~5/day (~150/month), indicating organic growth
- **Weekly seasonality** — weekdays outperform weekends, typical for B2B or content sites
- **Campaign spikes** are clearly visible — 7 distinct events detected as anomalies
- **Bounce rate is declining** — suggesting improving content quality or better-targeted traffic
- **Organic search** dominates traffic — SEO investment appears to be paying off
- **Mobile traffic** equals desktop — mobile optimization is critical

## Limitations
- Synthetic data with simplified traffic model
- No page-level or content analysis
- No conversion or revenue data
- No geographic segmentation
- Traffic source is simplified (one per day vs real multi-channel)

## How to Improve This Project
- Add conversion funnel analysis
- Include page-level engagement metrics
- Build traffic forecasting models
- Add A/B test result integration
- Include geographic and demographic breakdowns

## Production Considerations
- Real-time traffic monitoring dashboards
- Automated anomaly detection alerts
- Weekly/monthly traffic reports with trend decomposition
- Campaign attribution and ROI tracking

## Common Mistakes
- Comparing campaign days to average days without removing the campaign effect
- Ignoring day-of-week effects in week-over-week comparisons
- Treating bounce rate improvements as purely content-driven (could be traffic mix change)
- Not distinguishing organic growth from paid traffic growth

## Mini Challenge / Exercises
1. What is the month-over-month growth rate in sessions?
2. Remove campaign spike days and recalculate the average daily sessions.
3. Which day of the week has the highest pages-per-session?
4. Estimate what percentage of total yearly sessions come from campaign spikes.

## Final Summary / Key Takeaways
- Web traffic has clear trend, weekly seasonality, and campaign-driven spikes
- Decomposition separates organic growth from noise and events
- Anomaly detection identifies campaign effects for proper attribution
- Bounce rate improvement signals better content-traffic alignment
- Understanding traffic components is prerequisite to accurate forecasting